# 01 Problem Setup

This notebook defines the exact Lorenz-1960 initial value problem used for the FYDP numerical baseline. The goal is to freeze the equations, parameters, initial conditions, time interval, and numerical comparison grid before any solver-specific work begins.

## Research objective

Establish a trustworthy numerical baseline for the Lorenz-1960 ODE system that can later support ANN-based experiments. At this stage, the task is strictly numerical: define the benchmark correctly, solve it with a custom RK4 method and a standard scientific-library solver, and validate that both are consistent on the chosen interval.

## Verified local context used

- `lorenz_1960_equations_learning_guide.md`
- `paper1.txt` Section 4.2
- `fydp_research_structure_plan.md`

Those local sources agree on the benchmark setup:

- `k = 2`, `l = 1`
- `x(0) = 0.5`, `y(0) = 0.75`, `z(0) = 1`
- `t in [0, 1]`

## Lorenz-1960 equations

General form from the benchmark description:

\[
\frac{dx}{dt} = kl\left(\frac{1}{k^2+l^2}-\frac{1}{k^2}\right)yz,
\]
\[
\frac{dy}{dt} = kl\left(\frac{1}{l^2}-\frac{1}{k^2+l^2}\right)xz,
\]
\[
\frac{dz}{dt} = \frac{kl^2}{2}\left(\frac{1}{k^2}-\frac{1}{l^2}\right)xy.
\]

After substituting `k = 2` and `l = 1`, the system becomes:

\[
\frac{dx}{dt} = -0.1yz, \qquad
\frac{dy}{dt} = 1.6xz, \qquad
\frac{dz}{dt} = -0.75xy.
\]

## Numerical configuration used in this baseline

- Uniform comparison grid: 1001 time points on `[0, 1]`
- Primary fixed RK4 step: `h = 1e-3`
- Library baseline: `scipy.integrate.solve_ivp`
- High-accuracy library settings: `method="DOP853"`, `rtol=1e-10`, `atol=1e-12`

## Why these settings were chosen

- The local project notes require the Lorenz-1960 benchmark from `paper1` Section 4.2, not the appendix code fragment.
- The interval `[0, 1]` is short and is the exact interval used in the local project framing.
- A 1001-point grid aligns naturally with RK4 step size `1e-3`, which makes solver outputs directly comparable without hidden interpolation in the primary comparison.
- A high-accuracy SciPy solve is used as the numerical reference for validation, but it is still treated as a numerical approximation rather than analytical truth.


In [ ]:
import numpy as np
import pandas as pd

from lorenz1960_baseline import (
    DEFAULT_CONFIG,
    final_state_table,
    lorenz1960_coefficients,
    make_uniform_grid,
)


## Benchmark constants

The next cell makes every constant explicit so the downstream notebooks can be checked against the same setup.

In [ ]:
config = DEFAULT_CONFIG
coefficients = lorenz1960_coefficients(config.k, config.l)
time_grid = make_uniform_grid(config.t_span, step=config.rk4_step)

problem_summary = pd.DataFrame(
    {
        "item": [
            "k",
            "l",
            "initial_state",
            "time_interval",
            "rk4_step",
            "n_eval_points",
            "scipy_method",
            "scipy_rtol",
            "scipy_atol",
            "coeff_dx",
            "coeff_dy",
            "coeff_dz",
        ],
        "value": [
            config.k,
            config.l,
            config.initial_state,
            config.t_span,
            config.rk4_step,
            config.n_eval,
            config.scipy_method,
            config.scipy_rtol,
            config.scipy_atol,
            coefficients[0],
            coefficients[1],
            coefficients[2],
        ],
    }
)
problem_summary


## Discretization check

This confirms that the chosen RK4 step size covers the interval exactly and produces the shared comparison grid.

In [ ]:
pd.DataFrame(
    {
        "quantity": ["t_start", "t_end", "grid_points", "time_step"],
        "value": [time_grid[0], time_grid[-1], len(time_grid), time_grid[1] - time_grid[0]],
    }
)


## Libraries used

- `numpy` for vectorized numerical work
- `pandas` for small summary tables
- `matplotlib` for plotting in later notebooks
- `scipy.integrate.solve_ivp` for the scientific-library baseline

## Assumptions separated from verified facts

Verified from local project files:

- The Lorenz-1960 equation form
- Parameter values `k = 2`, `l = 1`
- Initial conditions `(0.5, 0.75, 1.0)`
- Time interval `[0, 1]`

Chosen numerical assumptions for this baseline:

- Fixed RK4 step `h = 1e-3`
- High-accuracy SciPy baseline using `DOP853`
- Output comparison on the same 1001-point grid

These choices are standard and simple, but they are still numerical design decisions rather than facts from the paper.